# IBM Quantum Runtime: Primitivos
## Una capacidad exclusiva de la Plataforma IBM Quantum

**Por joseluisrg (con asistencia de IA)**

Las computadoras cuánticas son ruidosas. A diferencia de las computadoras clásicas, cada compuerta que aplicas tiene una pequeña probabilidad de introducir un error, y los qubits pierden su estado cuántico con el tiempo simplemente por estar inactivos. La Plataforma IBM Quantum aborda esto con un conjunto de herramientas integradas directamente en su capa de ejecución en la nube — herramientas que no se pueden obtener de un SDK cuántico genérico ni de un simulador local.

**¿Qué hace única a la plataforma de IBM Quantum?**

- **Supresión de errores integrada** (Desacoplamiento Dinámico) — evita que los errores se acumulen mientras los qubits esperan
- **Mitigación de errores integrada** (Twirling de Compuertas, Extrapolación de Ruido Cero) — corrige errores estadísticamente después de la ejecución
- **Modos de ejecución** (`Job`, `Session`, `Batch`) — controla cómo se programan tus trabajos en hardware cuántico real
- **Transpilación automática** — reescribe tu circuito en las instrucciones nativas que entiende el hardware

Este cuaderno demuestra estas capacidades a través de dos **Primitivos de Qiskit Runtime** aplicadas a un circuito de estado Bell:
1. `SamplerV2` — *"Ejecuta el circuito y cuenta lo que sale"*
2. `EstimatorV2` — *"Ejecuta el circuito y calcula el valor promedio de una cantidad física"*

Por defecto se ejecuta localmente (sin necesidad de cuenta), e incluye instrucciones para cambiar a hardware real de IBM Quantum.

---
**Referencias:**
- [Documentación de Primitivos IBM Quantum](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-primitives)
- [Comenzar con primitivos](https://docs.quantum.ibm.com/guides/get-started-with-primitives)
- [API SamplerV2](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/sampler-v2)

## 0. Instalar dependencias

Ejecuta esta celda una sola vez para instalar los paquetes necesarios.

In [ ]:
# Descomenta y ejecuta para instalar
# %pip install qiskit qiskit-ibm-runtime matplotlib pylatexenc

## 1. Construir el Circuito Cuántico — Estado Bell

Preparamos el estado Bell maximalmente entrelazado:  
$$|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$$

Este es el "Hola Mundo" del entrelazamiento cuántico. Al medirlos, ambos qubits siempre coinciden — ya sea ambos `0` o ambos `1`. Cualquier desviación de este patrón en hardware real es una señal directa de ruido.

In [ ]:
from qiskit import QuantumCircuit

# ── Circuito del estado Bell ────────────────────────────────────────────────
bell = QuantumCircuit(2)
bell.h(0)          # Hadamard: pone el qubit 0 en superposición
bell.cx(0, 1)      # CNOT: entrelaza el qubit 1 con el qubit 0

# Dibuja el circuito
bell.draw('mpl', style='iqp')

## 2. Elegir el backend de ejecución

**Opción A (por defecto):** Ejecutar localmente con simuladores de vector de estado ideales — sin necesidad de cuenta.  
**Opción B:** Ejecutar en hardware real de IBM Quantum usando Qiskit Runtime.

> 💡 Para usar la Opción B, crea una cuenta gratuita en [quantum.ibm.com](https://quantum.ibm.com) y obtén tu token de API.

---

### 📖 Conceptos clave en esta celda

**`generate_preset_pass_manager(backend, optimization_level)`**  
Tu circuito está escrito en compuertas abstractas de Qiskit como `H` (Hadamard) y `CX` (CNOT). Los procesadores cuánticos reales de IBM no entienden esas compuertas — cada chip tiene su propio pequeño conjunto de operaciones *nativas* que puede ejecutar físicamente (por ejemplo, `SX`, `RZ` y `ECR` en muchos dispositivos IBM). Un *pass manager* es una cadena de pasos de compilación que traduce tu circuito abstracto a uno que el hardware puede ejecutar. Piénsalo como un compilador que convierte código fuente de alto nivel en código máquina.

`optimization_level=1` es una configuración equilibrada: optimiza el circuito lo suficiente para reducir errores sin tardar demasiado en compilar. Los niveles van de `0` (sin optimización, más rápido) a `3` (máxima optimización, más lento).

---

**`SamplerV2(mode=backend)` vs `EstimatorV2(mode=backend)`**  
Estas son las dos *primitivas* — los bloques de construcción fundamentales para ejecutar circuitos en hardware IBM. Comparten la misma interfaz `mode=backend` pero responden preguntas diferentes:

| Primitiva | Pregunta que responde | Salida | Uso típico |
|---|---|---|---|
| `SamplerV2` | *"¿Qué cadenas de bits obtengo y con qué frecuencia?"* | Conteos / probabilidades para cada resultado de medición | Estudiar distribuciones de salida, algoritmos de muestreo cuántico, verificar corrección de circuitos |
| `EstimatorV2` | *"¿Cuál es el valor promedio de esta cantidad física?"* | Un único número (valor esperado) por observable | Algoritmos variacionales (VQE, QAOA), cálculo de niveles de energía, medición de correlaciones |

Una analogía útil: el Sampler es como tirar un dado 1000 veces y registrar cada resultado. El Estimator es como preguntar *"¿cuál es el resultado promedio?"* — omite los resultados brutos y te da directamente el estadístico resumen, con control de precisión estadística integrado.

---

**`sampler.options.dynamical_decoupling.enable = True`**  
Los qubits son frágiles. Incluso cuando un qubit simplemente está *esperando* (sin que se le aplique ninguna compuerta), es constantemente perturbado por su entorno — qubits cercanos, ruido electromagnético, fluctuaciones térmicas. Esta deriva se llama *decoherencia*, y causa errores.

El *Desacoplamiento Dinámico* (DD) combate la decoherencia manteniendo los qubits inactivos ocupados con pulsos cuidadosamente cronometrados que cancelan las perturbaciones ambientales. Los pulsos están diseñados de modo que su efecto neto sobre el qubit sea cero — el qubit termina en el mismo estado en que empezó — pero el ruido que cancelan no regresa. Es análogo a los audífonos con cancelación de ruido: los audífonos emiten sonido que interfiere destructivamente con el ruido ambiental.

Esto ocurre completamente a nivel de pulsos del hardware, de forma invisible para la lógica de tu circuito.

---

**`sampler.options.dynamical_decoupling.sequence_type = "XpXm"`**  
No todas las secuencias de pulsos DD son igualmente efectivas. Diferentes secuencias cancelan diferentes tipos de ruido:

| Secuencia | Descripción | Mejor para |
|---|---|---|
| `"XX"` | Dos pulsos X, secuencia más simple | Uso general, bajo costo adicional |
| `"XpXm"` | Pulso X-positivo seguido de X-negativo | Circuitos con compuertas de dos qubits; cancela componentes de ruido Z y X |
| `"XY4"` | Secuencia alterna de cuatro pulsos X/Y | Supresión más fuerte; tiempos de inactividad más largos |

`"XpXm"` es un buen valor predeterminado para circuitos entrelazados como el nuestro, porque maneja los tipos de ruido más comunes alrededor de las compuertas de dos qubits (como el CNOT/CX que usamos).

---

**`sampler.options.twirling.enable_gates = True`**  
El ruido en las compuertas cuánticas es complicado porque puede ser *coherente* — lo que significa que los errores se acumulan sistemáticamente en lugar de aleatoriamente, haciéndolos mucho más difíciles de corregir. Imagina que cada compuerta sobre-rota 1°: después de 100 compuertas estás desviado 100°, no solo un poco de ruido aleatorio.

El *Twirling de Compuertas* soluciona esto envolviendo aleatoriamente cada compuerta de dos qubits con compuertas de Pauli (X, Y, Z) adicionales antes y después. Estos envoltorios aleatorios no cambian la operación deseada (se cancelan matemáticamente), pero *dispersan* cualquier error coherente en ruido aleatorio. El ruido aleatorio es mucho más fácil de modelar y mitigar que la deriva sistemática.

El resultado: el ruido de tu circuito se vuelve más predecible, lo que hace que la mitigación de errores posterior (como la Extrapolación de Ruido Cero) sea más efectiva.

In [ ]:
USE_REAL_HARDWARE = True   # ← Cambia a True para ejecutar en hardware IBM Quantum
IBM_QUANTUM_TOKEN = ""    # ← Pega tu token de API aquí si usas hardware real

if USE_REAL_HARDWARE:
    # ── Opción B: IBM Quantum Platform (Qiskit Runtime) ──────────────────────
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2, EstimatorV2
    from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

    service = QiskitRuntimeService(
        channel="ibm_quantum_platform",
        token=IBM_QUANTUM_TOKEN
    )
    backend = service.least_busy(operational=True, simulator=False)
    print(f"✅ Usando backend real: {backend.name} ({backend.num_qubits} qubits)")

    # Traduce compuertas abstractas de Qiskit (H, CX) a instrucciones nativas del hardware.
    # optimization_level=1: equilibrio entre tiempo de compilación y calidad del circuito.
    pm = generate_preset_pass_manager(backend=backend, optimization_level=1)

    # CLAVE: añadir mediciones al circuito *lógico* ANTES de la transpilación.
    # Si llamas measure_all() sobre el circuito ISA ya transpilado, medirá
    # TODOS los qubits físicos del dispositivo (p. ej. 127), produciendo cadenas
    # de bits enormes que entierran los resultados de 2 qubits del estado Bell.
    bell_measured = bell.copy()
    bell_measured.measure_all()              # mide solo los qubits lógicos 0 y 1
    sampler_circuit = pm.run(bell_measured)  # el transpilador mapea las mediciones correctamente
    isa_circuit     = pm.run(bell.copy())    # versión sin mediciones, para EstimatorV2

    # SamplerV2: ejecuta circuitos y devuelve conteos / probabilidades de medición.
    # EstimatorV2: ejecuta circuitos y devuelve valores de expectación de observables.
    # Ambos usan mode=backend para apuntar a la misma máquina física.
    sampler   = SamplerV2(mode=backend)
    estimator = EstimatorV2(mode=backend)

    # ── Supresión de errores: Desacoplamiento Dinámico ───────────────────────
    # Inserta pulsos de cancelación durante el tiempo de inactividad del qubit.
    # Secuencia XpXm: efectiva para circuitos con compuertas de dos qubits.
    sampler.options.dynamical_decoupling.enable = True
    sampler.options.dynamical_decoupling.sequence_type = "XpXm"
    print("🛡️  Desacoplamiento Dinámico activado (secuencia XpXm)")

    # ── Mitigación de errores: Twirling de Compuertas ────────────────────────
    # Aleatoriza los errores coherentes de las compuertas en ruido depolarizante,
    # haciendo los errores más fáciles de modelar y corregir en post-procesamiento.
    sampler.options.twirling.enable_gates = True
    sampler.options.twirling.num_randomizations = "auto"
    print("🔀 Twirling de Compuertas activado")

else:
    # ── Opción A: Simuladores locales de vector de estado ideal ──────────────
    from qiskit.primitives import StatevectorSampler as SamplerV2
    from qiskit.primitives import StatevectorEstimator as EstimatorV2

    bell_measured = bell.copy()
    bell_measured.measure_all()
    sampler_circuit = bell_measured   # el simulador local acepta compuertas estándar
    isa_circuit     = bell.copy()     # sin mediciones, para EstimatorV2
    sampler   = SamplerV2()
    estimator = EstimatorV2()
    print("🖥️  Ejecutando localmente con simuladores de vector de estado ideales")
    print("   (Cambia USE_REAL_HARDWARE = True para ejecutar en hardware IBM Quantum)")

## 3. Primitiva 1: `SamplerV2` — Muestrear la Distribución de Salida

El **Sampler** responde la pregunta: *"Si ejecuto este circuito y mido los qubits, ¿qué resultados obtengo?"*

Debido a que la mecánica cuántica es probabilística, no podemos saber el resultado de una sola ejecución de antemano. En su lugar, ejecutamos el circuito muchas veces — cada ejecución se llama un *shot* — y recopilamos estadísticas. El Sampler devuelve el conteo de cada cadena de bits única observada en todos los shots.

Para el estado Bell, los únicos resultados válidos son `00` y `11` (ambos qubits correlacionados). En un simulador perfecto aparecen con exactamente 50% de probabilidad cada uno. En hardware real también verás pequeñas cantidades de `01` y `10` — esos son errores.

In [ ]:
import matplotlib.pyplot as plt

# sampler_circuit ya incluye mediciones (añadidas antes de la transpilación en la Celda 2).
# NO llamar measure_all() aquí — en hardware real mediría todos los qubits
# físicos y produciría cadenas de bits de 127 caracteres sin sentido.

# ── Ejecutar la primitiva Sampler ──────────────────────────────────────────
NUM_SHOTS = 1024

sampler_job = sampler.run([sampler_circuit], shots=NUM_SHOTS)
sampler_result = sampler_job.result()

# Extraer conteos — el registro se llama 'meas' por measure_all()
pub_result = sampler_result[0]
counts = pub_result.data.meas.get_counts()

# ── Resumen en consola ─────────────────────────────────────────────────────
print(f"📊 Resultados de medición ({NUM_SHOTS} shots) — {len(counts)} cadenas de bits únicas:")
for bitstring, count in sorted(counts.items(), key=lambda x: -x[1])[:20]:
    prob = count / NUM_SHOTS
    bar  = '█' * int(prob * 40)
    print(f"   |{bitstring}⟩  {bar:<40}  {count:4d} ({prob:.1%})")

# ── Gráfica: barras horizontales — escala limpiamente con muchas cadenas ────
sorted_counts = sorted(counts.items(), key=lambda x: -x[1])[:16]
labels = [f"|{k}⟩" for k, _ in sorted_counts]
probs  = [v / NUM_SHOTS for _, v in sorted_counts]

fig_h = max(3.5, len(labels) * 0.52)
fig, ax = plt.subplots(figsize=(7, fig_h))

bars = ax.barh(labels[::-1], probs[::-1], color='#6929c4', alpha=0.88, height=0.6)
ax.bar_label(bars, labels=[f'{p:.1%}' for p in probs[::-1]], padding=5, fontsize=10)
ax.set_xlabel('Probabilidad', fontsize=12)
ax.set_xlim(0, max(probs) * 1.3)
title = f'Estado Bell — Salida SamplerV2 ({NUM_SHOTS} shots)'
if len(counts) > 16:
    title += f'  [top {len(labels)} de {len(counts)}]'
ax.set_title(title, fontsize=13, fontweight='bold')
ax.spines[['top', 'right']].set_visible(False)
fig.subplots_adjust(left=0.14, right=0.93, top=0.90, bottom=0.12)
plt.show()

**Resultado esperado:** Solo `00` y `11` aparecen con ~50% de probabilidad cada uno.  
Cualquier conteo de `01` o `10` es ruido del hardware — las opciones de supresión de errores de la Celda 2 ayudan a reducirlos.

---

## 4. Primitiva 2: `EstimatorV2` — Calcular Valores Esperados

El **Estimator** responde una pregunta diferente: *"¿Cuál es el valor promedio de un observable físico en este estado cuántico?"*

En mecánica cuántica, un *observable* es cualquier cosa que puedas medir — por ejemplo, el espín de un qubit a lo largo del eje Z. El *valor de expectación* $\langle O \rangle$ es el resultado promedio que obtendrías si midieras ese observable muchas veces. Matemáticamente:

$$\langle O \rangle = \langle \psi | O | \psi \rangle$$

donde $|\psi\rangle$ es el estado cuántico preparado por tu circuito.

Usamos *operadores de Pauli* (X, Y, Z) como observables. Un qubit individual medido a lo largo de Z da `+1` si está en el estado `|0⟩` y `-1` si está en el estado `|1⟩`. Los operadores de múltiples qubits como `ZZ` dan el *producto* de ambas mediciones.

Para nuestro estado Bell medimos tres observables:
- **`ZZ`** — ¿están los dos qubits correlacionados en Z? Valor ideal: `+1.0` (siempre coinciden)
- **`XX`** — ¿están correlacionados en X? Valor ideal: `+1.0` (también siempre coinciden, firma del entrelazamiento)
- **`ZI`** — ¿qué es el qubit 0 solo en Z? Valor ideal: `0.0` (maximalmente incierto por sí solo)

In [ ]:
from qiskit.quantum_info import SparsePauliOp

# ── Usar el circuito transpilado ISA (obligatorio para hardware real IBM) ───
# EstimatorV2 valida que los circuitos solo contengan compuertas nativas del backend.
# bell.copy() lanzaría IBMInputValueError: 'h on qubits (0,) is not supported'
estimator_circuit = isa_circuit   # ya transpilado en la Celda 2

# ── Definir observables y reasignarlos al layout de qubits transpilado ──────
observables_logical = [
    SparsePauliOp("ZZ"),   # Medición Z correlacionada — ideal: +1.0
    SparsePauliOp("XX"),   # Medición X correlacionada — ideal: +1.0
    SparsePauliOp("ZI"),   # Z de un solo qubit en qubit 0 — ideal:  0.0
]
observable_labels = ["ZZ", "XX", "ZI"]
ideal_values      = [+1.0, +1.0,  0.0]

# obs.apply_layout(layout): tras la transpilación, los qubits lógicos se asignan
# a qubits físicos específicos del chip (p. ej. qubit lógico 0 → qubit físico 14).
# Sin esta reasignación, el observable ZZ apuntaría a los qubits físicos incorrectos
# y el estimador calcularía valores de expectación erróneos.
# apply_layout() lee el mapa de asignación de qubits del transpilador y reescribe
# las cadenas de Pauli para que coincidan — así ZZ en lógico (0,1) se convierte en
# ZZ en los qubits físicos reales que usa el circuito.
layout = estimator_circuit.layout
observables = [
    obs.apply_layout(layout) if layout is not None else obs
    for obs in observables_logical
]

# ── Ejecutar la primitiva Estimator ────────────────────────────────────────
# Cada PUB (Primitive Unified Bloc) es un par (circuito, observable).
# El Estimator ejecuta los tres en un único trabajo.
pubs = [(estimator_circuit, obs) for obs in observables]

estimator_job = estimator.run(pubs)
estimator_result = estimator_job.result()

print("📐 Valores esperados en el estado Bell |Φ+⟩:\n")
print(f"  {'Observable':<12} {'Medido':>10} {'Ideal':>8} {'Error':>8}")
print("  " + "-" * 42)

measured_evs = []
for i, (label, ideal) in enumerate(zip(observable_labels, ideal_values)):
    ev = float(estimator_result[i].data.evs)
    measured_evs.append(ev)
    error = abs(ev - ideal)
    print(f"  ⟨{label}⟩{'':<10} {ev:>+10.4f} {ideal:>+8.1f} {error:>8.4f}")

## 5. Visualizar los Valores Esperados

In [ ]:
import numpy as np

x = np.arange(len(observable_labels))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 5))

bars_measured = ax.bar(x - width/2, measured_evs, width,
                       label='Medido', color='#6929c4', alpha=0.9)
bars_ideal    = ax.bar(x + width/2, ideal_values, width,
                       label='Ideal (sin ruido)', color='#009d9a', alpha=0.9)

ax.set_xlabel('Observable', fontsize=13)
ax.set_ylabel('Valor de Expectación', fontsize=13)
ax.set_title('EstimatorV2 — Valores de Expectación del Estado Bell', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'⟨{l}⟩' for l in observable_labels], fontsize=12)
ax.set_ylim(-1.4, 1.4)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.legend(fontsize=11)
ax.bar_label(bars_measured, fmt='%.3f', padding=4, fontsize=10)
ax.bar_label(bars_ideal,    fmt='%.1f',  padding=4, fontsize=10)

fig.tight_layout()
plt.show()

## 6. Resumen — ¿Qué hace únicas a las Primitivas de Runtime?

| Característica | Simulador Local | IBM Quantum Runtime |
|---|---|---|
| API `SamplerV2` / `EstimatorV2` | ✅ Misma interfaz | ✅ Misma interfaz |
| Hardware cuántico real | ❌ | ✅ |
| Desacoplamiento Dinámico | ❌ | ✅ |
| Twirling de Compuertas | ❌ | ✅ |
| Extrapolación de Ruido Cero (ZNE) | ❌ | ✅ |
| Modos de ejecución Session / Batch | ❌ | ✅ |
| Transpilación automática para hardware | ❌ | ✅ |

---

### 🔬 ¿Por qué necesitamos supresión y mitigación de errores?

Las computadoras cuánticas actuales son dispositivos *cuánticos de escala intermedia con ruido* (NISQ, por sus siglas en inglés). Cada operación — cada compuerta, cada momento en que un qubit espera, cada medición — introduce una pequeña probabilidad de error. Para circuitos cortos esto es manejable, pero a medida que los circuitos se vuelven más profundos (más compuertas) o más anchos (más qubits), los errores se multiplican y pueden borrar por completo la señal cuántica.

IBM Quantum Runtime ofrece dos estrategias complementarias para combatir esto:

**Supresión de Errores** (ocurre *durante* la ejecución en el hardware)  
Técnicas como el Desacoplamiento Dinámico y las compuertas eficientes en pulsos reducen la tasa a la que ocurren errores en primer lugar, antes de que siquiera observes los resultados.

**Mitigación de Errores** (ocurre *después* de la ejecución, en post-procesamiento clásico)  
Incluso después de la supresión, algunos errores persisten. Las técnicas de mitigación usan ejecuciones adicionales y estadísticas inteligentes para estimar cuál habría sido la respuesta *ideal* y acercar los resultados a ella. No eliminan los errores — los compensan matemáticamente.

El beneficio: resultados en hardware real mucho más cercanos al ideal, sin necesidad de una computadora cuántica tolerante a fallos.

---

### 📐 Extrapolación de Ruido Cero (ZNE)

ZNE es una técnica de mitigación de errores disponible a través de `EstimatorV2` cuando configuras `estimator.options.resilience_level = 2`. La idea es contraintuitiva pero poderosa:

1. **Incrementar intencionalmente el ruido** en tu circuito estirando o repitiendo compuertas (esto amplifica los errores de forma controlada)
2. **Ejecutar el circuito a varios niveles de ruido** — p. ej. 1×, 2× y 3× el ruido original
3. **Extrapolar de vuelta a ruido cero** usando la tendencia de esos puntos de datos

Es como medir la temperatura de una varilla en varios puntos a lo largo de su longitud y extrapolar para predecir cuál sería en el extremo — nunca mides ruido cero directamente, pero puedes inferirlo matemáticamente.

**¿Cuándo es más útil ZNE?** Para calcular valores de expectación con `EstimatorV2`, especialmente en algoritmos variacionales (VQE, QAOA) donde necesitas estimaciones precisas de energía. Es menos útil para el Sampler, ya que trabaja con valores de expectación en lugar de conteos brutos.

```python
# Activar ZNE en el Estimator (resilience_level 2)
estimator.options.resilience_level = 2
```

---

### ⏱️ Modos de Ejecución: Session y Batch

Las computadoras cuánticas de IBM son recursos compartidos — muchos usuarios ponen en cola trabajos en la misma máquina. La forma en que se *programan* tus trabajos tiene un gran impacto en el rendimiento. IBM Quantum Runtime ofrece tres modos de ejecución:

**Modo Job** (el que usa este cuaderno por defecto)  
Cada llamada a `sampler.run()` o `estimator.run()` es un trabajo independiente, puesto en cola y programado por separado. Simple y adecuado para experimentos puntuales.

**Modo Session** — para algoritmos *iterativos*  
Una Session reserva un *bloque de tiempo dedicado* en un backend específico. Todos los trabajos enviados dentro de la sesión se ejecutan consecutivamente con un tiempo de espera mínimo entre ellos. Esto es esencial para algoritmos como VQE o QAOA, que ejecutan cientos de evaluaciones de circuitos en un ciclo de retroalimentación (el resultado de una ejecución informa los parámetros de la siguiente). Sin una sesión, cada uno de esos cientos de trabajos esperaría en la cola general, potencialmente sumando horas de tiempo total de ejecución.

```python
from qiskit_ibm_runtime import Session
with Session(backend=backend) as session:
    estimator = EstimatorV2(mode=session)
    # todas las llamadas a estimator.run() aquí comparten tiempo dedicado en la QPU
```

**Modo Batch** — para cargas de trabajo *paralelas*  
Un Batch agrupa muchos circuitos independientes y los envía como un único bloque al programador. Los trabajos aún se ejecutan de forma independiente (sin estado compartido entre ellos), pero obtienen prioridad de programación como grupo. Ideal para barridos de parámetros, experimentos de caracterización de ruido, o cualquier momento en que tengas muchos circuitos que ejecutar y que no dependen entre sí.

```python
from qiskit_ibm_runtime import Batch
with Batch(backend=backend) as batch:
    sampler = SamplerV2(mode=batch)
    # todas las llamadas a sampler.run() aquí se programan como grupo
```

---

La ventaja clave de la Plataforma IBM es que **la misma interfaz simple de `sampler.run()` / `estimator.run()` aplica de forma transparente todas estas capas de experiencia en hardware** — transpilación, supresión de errores, mitigación de errores y programación inteligente — que de otro modo requerirían meses de conocimiento de bajo nivel para implementar manualmente.

### Próximos pasos
- [Activar el modo Session](https://quantum.cloud.ibm.com/docs/en/guides/sessions) para algoritmos iterativos (VQE, QAOA)
- [Explorar opciones de mitigación de errores](https://quantum.cloud.ibm.com/docs/en/guides/configure-error-mitigation)
- [Probar Qiskit Functions](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-functions) para bloques de construcción de aplicaciones de nivel superior